# TSFM-Interp quickstart

Compare two small time-series foundation models (Chronos-Bolt-small and Chronos-2) on a small synthetic corpus and end with one interactive HTML report. About 25 minutes on one GPU (budget up to an hour on a slower one) (set the runtime to a GPU: Runtime > Change runtime type).

What you get: behavioural accuracy per family, representation similarity, cross-model stitching, corruption sensitivity and activation patching, attention structure, sparse-autoencoder concepts and cross-model concept transfer.

What you do not get: held-out confirmation. That needs a sealed private split nobody has looked at, so the `confirm` stage is off and everything here is exploratory.

## 1. Install
Clones the repository (the configs live in it) and installs both packages, the lens from its subdirectory. Set the environment variable `QUICKSTART_REPO` to an existing checkout to skip the clone and install.

In [ ]:
import os, subprocess, sys
from pathlib import Path

def sh(cmd, cwd=None, env=None):
    """Run a shell command, streaming its output; raise if it fails."""
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")

REPO_URL = "https://github.com/DanK10097110/TSFM-Interp.git"
if os.environ.get("QUICKSTART_REPO"):
    REPO = Path(os.environ["QUICKSTART_REPO"]).resolve()
else:
    REPO = Path("TSFM-Interp").resolve()
    if not REPO.exists():
        sh(f"git clone --depth 1 {REPO_URL} {REPO}")
    sh(f"{sys.executable} -m pip install -q -e . -e tsfm_lens chronos-forecasting", cwd=REPO)
print("repo:", REPO)

## 2. Build the small corpus
400 synthetic series (two families), no downloads, about a minute. `--references none` skips the real-data leakage gate, which is a no-op for purely synthetic data; the build says so and records it in the manifest.

In [ ]:
sh(f"{sys.executable} tsfm_benchmark/example_runs/run_full.py "
   "--config tsfm_benchmark/configs/quickstart.yaml --out ./benchmark_quickstart --references none",
   cwd=REPO, env={"PYTHONPATH": "."})

## 3. Preflight
Checks the GPU, the corpus seal, each model's alignment and the statistical budget before spending GPU time.

In [ ]:
LENS = REPO / "tsfm_lens"
THREADS = {"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4", "OPENBLAS_NUM_THREADS": "4"}
sh(f"{sys.executable} run.py --config configs/quickstart.yaml --doctor", cwd=LENS, env=THREADS)

## 4. Run
All stages for the pair: about 25 minutes on one RTX A5000-class GPU. Finished stages are skipped on a rerun, so if Colab disconnects, rerun this cell.

In [ ]:
sh(f"{sys.executable} run.py --config configs/quickstart.yaml", cwd=LENS, env=THREADS)

## 5. Open the report
One self-contained HTML file.

In [ ]:
from IPython.display import HTML, IFrame, display
REPORT = LENS / "runs" / "quickstart" / "report.html"
print(REPORT, f"{REPORT.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(str(REPORT))
except ImportError:
    display(IFrame(os.path.relpath(REPORT), width="100%", height=700))

## 6. How to read it

Start at the scorecard at the top of the report. Every row says which rule produced its verdict, and a missing reference reads "not comparable" rather than a guess. Each claim in the report carries an **evidence class**, and the classes are not interchangeable:

| Class | Where | What it tells you | Can it say "causal"? |
|---|---|---|---|
| Behavioural | L0 | Which model forecasts better, per family (MASE, sMAPE, pinball; paired bootstrap over series, Holm-corrected) | No, it is about outputs |
| Geometric | L1 | Whether two models' layers have similar representation shape (CKA, RSA) | No, correlational |
| Linear-translatable | L2 | Whether one model's layer linearly predicts another's, reported as gain over an input-feature baseline | No |
| Causal within one model | L3, concepts | Corruption fingerprints, activation patching, SAE feature ablation against a random-direction null | Yes, inside that model only. Activations are never moved between models |
| Held-out | confirm | Hypotheses registered on dev data, tested once on a sealed private split | **Not part of this quickstart** |

Read the report with these in mind:

- **The resampling unit is the series**, never windows, so confidence intervals are about series-to-series variation.
- **400 series gives wide intervals.** Expect many "not significant" verdicts and few or no modular concepts. That is the sample size, not a bug.
- **Absence of agreement is not disagreement.** "No specific agreement" between two models' concepts is not evidence that they differ.
- **Skipped analyses are rendered with their reason.** A section that says why it did not run is the pipeline working as designed.

The findings also exist as data in `report/findings.json`:

In [ ]:
import json
findings = json.loads((REPORT.parent / "report" / "findings.json").read_text(encoding="utf-8"))
items = findings if isinstance(findings, list) else findings.get("findings", [])
print(len(items), "findings")
for f in items[:15]:
    print(f"[{f.get('stage')}/{f.get('evidence_class')}] {f.get('plain') or f.get('text')}")

## Where next

- Swap a model: copy `tsfm_lens/configs/quickstart.yaml` and edit `models:`; see `tsfm_lens/docs/ADDING_A_MODEL.md`.
- The full pipeline (seven models, ~1000-series corpus, held-out confirmation): `tsfm_lens/configs/panel7_v2.yaml`; its published run is `examples/panel7_v2`.
- Every reduced knob in the quickstart config states the full value in an inline comment.